In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override=True)  # override=True 确保 .env 会覆盖系统已有的同名变量
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))  # 确认代理是否生效
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def get_user_info() -> str:
    """Look up information about the current user."""
    return "No user profile on file"

agent = create_agent(
    model = "openrouter:deepseek/deepseek-v4-flash-0731",
    tools = [get_user_info],
    checkpointer=InMemorySaver()
)
thread_config = {"configurable":{"thread_id":"DTEST001"}}
response = agent.invoke(
    {"messages": [{"role":"user","content":"Hi, My name is Bob"}]},
    thread_config
)["messages"][-1].content
print(response)
response = agent.invoke(
    {"messages":[{"role":"user","content":"Hi, What's my name"}]},
    thread_config
)["messages"][-1].content

print(response)

Nice to meet you, Bob! It looks like there isn't a user profile on file for you yet — that's totally okay. 

Is there anything I can help you with today? Whether it's answering a question, working through a problem, or just having a conversation, I'm here to help. What's on your mind?
Based on our conversation, you told me your name is **Bob**! 😊

Note that I don't have a stored user profile on file, so if you'd like me to remember your name going forward, you'll just need to remind me each time we chat. 

Is there anything else I can help you with, Bob?


In [7]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    preferences: dict

agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_user_info],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_id": "user_123",
        "preferences": {"theme": "dark"}
    },
    {"configurable": {"thread_id": "1"}})

print(result)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='dfa80f1a-3485-4e22-a490-34d7d5dc2716'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user greeted me. I should respond in a friendly way. Since they mentioned the user, maybe I should look up user info? Actually, the greeting doesn\'t require it. But let me consider - the tools include get_user_info. The user just said "Hello". I can greet them back. But maybe it\'s useful to look up user info to personalize. Let me do that since I have the tool available and it could help personalize the response.\n\nActually, on a simple greeting, it\'s fine to just greet back. But let me consider looking up user info to personalize. I\'ll call get_user_info to greet them properly.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user greeted me. I should respond in a friendly way. Since they mentioned the user, maybe I should look up user i

In [10]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str,Any] | None:
    """Keep only the last few messages to fit context window"""
    messages = state["messages"]
    if len(messages) <= 3:
        return None
    
    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages":[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }
agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is Bob! You told me at the very beginning of our chat. 😊

Would you like another poem, or is there something else you'd like to chat about, Bob?


In [12]:
# 删除置顶的消息
from langchain.messages import RemoveMessage
def delete_messages(state):
    messages = state["messages"]
    if len(messages) > 2:
        return {"messages":[RemoveMessage(id=m.id) for m in messages[:2]]}

# 删除全部消息
from langgraph.graph.message import REMOVE_ALL_MESSAGES
def delete_messages(state):
    return {"messages":[RemoveMessage(id=REMOVE_ALL_MESSAGES)]}


In [18]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent,AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from requests import delete

@after_model
def delete_old_messages(state:AgentState,runtime:Runtime)->dict | None:
    """Remove old messages to keep conversation manageabled."""
    messages = state["messages"]
    if len(messages)>2:
        # remove messages
        return {"messages":[RemoveMessage(id=m.id) for m in messages[:2]]}

agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    system_prompt="Please be concise and to the point.",
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
stream = agent.stream_events(
    {"messages":[{"role":"user","content":"hi,i'am bob"}]},
    config,
    version="v3"
)
for snapshot in stream.values:
    print([(message.type,message.content) for message in snapshot["messages"]])

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "write a short poem about cats"}]},
    config,
    version="v3",
)
for snapshot in stream.values:
    print([(message.type, message.content) for message in snapshot["messages"]])

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "what's my name?"}]},
    config,
    version="v3",
)
for snapshot in stream.values:
    print([(message.type, message.content) for message in snapshot["messages"]])



[('human', "hi,i'am bob")]
[('human', "hi,i'am bob"), ('ai', [{'type': 'reasoning', 'reasoning': 'We need answer user. Need infer context? "hi,i\'am bob" Probably greeting. Need respond friendly. Maybe mention name? "Hi Bob! How can I help you today?" Ensure grammar maybe "I\'m".', 'index': 0}, {'type': 'text', 'text': 'Hi Bob! Nice to meet you. How can I help you today?', 'index': 1}])]
[('human', "hi,i'am bob"), ('ai', [{'type': 'reasoning', 'reasoning': 'We need answer user. Need infer context? "hi,i\'am bob" Probably greeting. Need respond friendly. Maybe mention name? "Hi Bob! How can I help you today?" Ensure grammar maybe "I\'m".', 'index': 0}, {'type': 'text', 'text': 'Hi Bob! Nice to meet you. How can I help you today?', 'index': 1}]), ('human', 'write a short poem about cats')]
[('human', "hi,i'am bob"), ('ai', [{'type': 'reasoning', 'reasoning': 'We need answer user. Need infer context? "hi,i\'am bob" Probably greeting. Need respond friendly. Maybe mention name? "Hi Bob! How

In [ ]:
# langchain 的自动摘要
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

checkpoints = InMemorySaver()
agent = create_agent(
    model = "openrouter:deepseek/deepseek-v4-flash-0731",
    tools = [],
    middleware = [
        SummarizationMiddleware(
            model = "openrouter:deepseek/deepseek-v4-flash-0731",
            trigger=("tokens",4000),
            keep=("messages",20)
        )
    ],
    checkpointer=checkpoints
)
config:RunnableConfig={"configurable":{"thread_id":"DTEST001"}}
agent.invoke({"messages":"hi, my name is bob"},config)
agent.invoke({"messages":"write a short poem about cats"},config)
agent.invoke({"messages":"now do the same but for dogs"},config)
finnal_response = agent.invoke({"messages":"what's my name?"},config)

finnal_response["messages"][-1].pretty_print()



================================== Ai Message ==================================

Your name is Bob! You told me at the very start of our chat. 😊 

Do you go by any nicknames, or is Bob the one and only?


In [ ]:
# tool拿到短期记忆
from langchain.agents import create_agent,AgentState
from langchain.tools import tool,ToolRuntime

class CustomState(AgentState):
    user_id:str

@tool
def get_user_info(runtime:ToolRuntime) -> str:
    """Look up user info """
    user_id = runtime.state["user_id"]
    return "User is John Wick" if user_id == "DTEST001" else "Unknown user"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_user_info],
    state_schema=CustomAgentState,
)
result = agent.invoke(
    {
        "messages":"look up user information",
        "user_id":"DTEST001"
    }
)
print(result["messages"][-1].content)

The user information has been retrieved. The user is **John Wick**.


In [28]:
# 从tools中拿到短期记忆
from langchain.tools import tool,ToolRuntime
from langchain_core.runnables import RunnableConfig
from langchain.messages import ToolMessage
from langchain.agents import create_agent,AgentState
from langgraph.types import Command
from pydantic import BaseModel

class CustomState(AgentState):
    user_name:str
class CustomContext(BaseModel):
    user_id:str

@tool 
def update_user_info(runtime:ToolRuntime[CustomContext,CustomState])->Command:
    """Look up and update user info."""
    user_id = runtime.context.user_id
    name = "John wick" if user_id == "DTEST001" else "Unknown user"
    return Command(update={
        "user_name":name,
        "messages":[
            ToolMessage(
                "Successfully looked up user information",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })

@tool 
def greet(runtime:ToolRuntime[CustomContext,CustomState]) ->str | Command:
    """Use this to greet the user once you found their info """
    user_name = runtime.state.get("username",None)
    if user_name is None:
        return Command(update={
            "messages":[
                ToolMessage("Please call the update_user_info tool it will get and used",tool_call_id=runtime.tool_call_id)
            ]
        })
    return f"hello {user_name}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[update_user_info, greet],
    state_schema=CustomState,
    context_schema=CustomContext,
)

agent.invoke(
    {"messages": [{"role": "user", "content": "greet the user"}]},
    context=CustomContext(user_id="DTEST001"),
)

{'messages': [HumanMessage(content='greet the user', additional_kwargs={}, response_metadata={}, id='67bccbef-b54e-40e7-b199-a091250d1316'),
  AIMessage(content="I'll look up your info", additional_kwargs={'reasoning_content': 'The user wants me to greet them. First I need to find their info using update_user_info, then greet them. Let me call update_user_info first to find their info.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants me to greet them. First I need to find their info using update_user_info, then greet them. Let me call update_user_info first to find their info.'}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-0731', 'id': 'gen-1787296691-7a32LsNAsrA7TyJaQIJK', 'created': 1787296691, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 3.178e-05, 'cost_details': {'upstream_inference_completions_cost': 1.022e-05, 'upstream_inference_

In [30]:
from langchain.agents import create_agent
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class CustomContext(TypedDict):
    user_name: str


def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is always sunny!"


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context["user_name"]
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt


agent = create_agent(
    model= "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
    middleware=[dynamic_system_prompt],
    context_schema=CustomContext,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    context=CustomContext(user_name="John Smith"),
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the weather in SF?
================================== Ai Message ==================================

John Smith, let me check the weather in San Francisco for
Tool Calls:
  get_weather (call_79b7ebd8dfa44362baadee09)
 Call ID: call_79b7ebd8dfa44362baadee09
  Args:
    city: San Francisco
================================= Tool Message =================================
Name: get_weather

The weather in San Francisco is always sunny!
================================== Ai Message ==================================

John Smith, the weather in San Francisco is **always sunny**! ☀️


In [33]:
# 使用before model标记使得在llm处理前处理数据
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime)->dict[str,Any] | None:
    """keep only the last few messages to fit context window"""
    messages = state["messages"]
    if len(messages)<=3:
        return None
    first_msg = messages[0]
    recent_msg =messages[-3:] if len(messages)%2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_msg
    return {
        "messages":[
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }
agent = create_agent(
    "openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)
config:RunnableConfig = {"configurable":{"thread_id":"DTEST001"}}
agent.invoke({"messages":"hi,my name is bob"},config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)
final_response = agent.invoke({"messages": "what's my name?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Your name is **Bob**! 😊 

You told me right at the start of our chat. It's a great name—short, sweet, and easy to remember. Did you forget, or were you just testing me? 😄


In [ ]:
# after model在模型处理之后再进行处理
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime


@after_model
def validate_response(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove messages containing sensitive words."""
    STOP_WORDS = ["password", "secret"]
    last_message = state["messages"][-1]
    if any(word in last_message.content for word in STOP_WORDS):
        return {"messages": [RemoveMessage(id=last_message.id)]}
    return None

agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[validate_response],
    checkpointer=InMemorySaver(),
)